# Pythia-410m: the full 50M-token experiment

GaugeMLA beside its ablation A_p^ramp and the methods it is compared with (TransMLA, MHA2MLA, CARE). **GaugeMLA, our method, is the arm `mla_w64_ours_unrope_cost_kl`** (kits `p410m_klgc` at 50M and `p410m_kl5m` at 5M): each un-roped rotary plane is read out by its head's KL stand-in. `mla_w64_ours_unrope_cost` is **A_p^ramp, an ablation**: GaugeMLA without the KL stand-in, which un-ropes each plane by its average rotation over the offsets instead (the earlier version of the method). It is built and trained with the baselines, in `p410m_gc`. To run GaugeMLA alone, use `pythia410m_gaugemla.ipynb`.

Requires **Google Colab Pro+**. **Runtime: A100 GPU, High-RAM** (Runtime → Change runtime type). Run `pythia410m_init.ipynb` first.

Distils each branch toward Pythia-2.8b for 50M tokens, with the gradient clipped at 1.0. Rows go to `MyDrive/gaugemla/kl_standin_50m/p410m_klgc/runs/<stamp>/` (GaugeMLA) and `MyDrive/gaugemla/grad_clip/p410m_gc/runs/<stamp>/` (the rest). Each finished branch also saves its final model (fp32) under `models/`.

Each training cell has its own `BRANCH` (`<arm>_G<G>`; G = 4, 2, 1 is 25, 12.5, 6.25% of the KV cache): one branch, or empty for every branch listed in that cell, one after another. `TEST_TOKENS = 50000` is a quick test (about 7 optimiser steps) under its own stamp; 0 is the full run. A finished branch is skipped on a rerun.

Requires a Hugging Face token: add it as the Colab secret `HF_TOKEN` and give this notebook access.

In [ ]:
# @title Setup
# For a quick test set TEST_TOKENS = 50000 (about 7 optimiser steps per branch, under its own stamp);
# 0 is the full run, as in the paper.
TEST_TOKENS = 0  # @param {type:"integer"}

import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
# The repository is public. While it is private, uncomment these lines and add the Colab secret READ_MLA_TOKEN
# (a GitHub token that can read it):
# from google.colab import userdata
# os.environ["READ_MLA_TOKEN"] = userdata.get("READ_MLA_TOKEN")
# with open("/content/askpass.sh", "w") as f:
#     f.write('#!/bin/sh\necho "$READ_MLA_TOKEN"\n')
# os.chmod("/content/askpass.sh", 0o700)
# os.environ["GIT_ASKPASS"] = "/content/askpass.sh"
os.environ["GIT_TERMINAL_PROMPT"] = "0"
if os.path.isdir("/content/repo/.git"):
    subprocess.run(["git", "-C", "/content/repo", "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", "/content/repo", "reset", "--hard", "origin/main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                    "https://github.com/anon5159753/gaugemla.git", "/content/repo"], check=True)
    subprocess.run(["git", "-C", "/content/repo", "sparse-checkout", "set", "src", "colab"], check=True)
sys.path.insert(0, "/content/repo/colab")
import launch

In [ ]:
# @title GaugeMLA (our method): mla_w64_ours_unrope_cost_kl, 50M tokens (p410m_klgc)
BRANCH = "mla_w64_ours_unrope_cost_kl_G4"  # @param ["mla_w64_ours_unrope_cost_kl_G4", "mla_w64_ours_unrope_cost_kl_G2", "mla_w64_ours_unrope_cost_kl_G1", ""] {allow-input: true}
for branch in ([BRANCH] if BRANCH else ('mla_w64_ours_unrope_cost_kl_G4', 'mla_w64_ours_unrope_cost_kl_G2', 'mla_w64_ours_unrope_cost_kl_G1')):
    launch.run("p410m_klgc", "run", BRANCH=branch, TEST_TOKENS=TEST_TOKENS)

In [ ]:
# @title The baselines and A_p^ramp (ablation), 50M tokens (p410m_gc)
BRANCH = "mla_w64_transmla_sigma_G4"  # @param ["mla_w64_transmla_sigma_G4", "mla_w64_mha2mla_s2norm_G4", "mla_w64_care_sigma_G4", "mla_w64_ours_unrope_cost_G4", "mla_w64_transmla_sigma_G2", "mla_w64_mha2mla_s2norm_G2", "mla_w64_care_sigma_G2", "mla_w64_ours_unrope_cost_G2", "mla_w64_transmla_sigma_G1", "mla_w64_mha2mla_s2norm_G1", "mla_w64_care_sigma_G1", "mla_w64_ours_unrope_cost_G1", ""] {allow-input: true}
for branch in ([BRANCH] if BRANCH else ('mla_w64_transmla_sigma_G4', 'mla_w64_mha2mla_s2norm_G4', 'mla_w64_care_sigma_G4', 'mla_w64_ours_unrope_cost_G4', 'mla_w64_transmla_sigma_G2', 'mla_w64_mha2mla_s2norm_G2', 'mla_w64_care_sigma_G2', 'mla_w64_ours_unrope_cost_G2', 'mla_w64_transmla_sigma_G1', 'mla_w64_mha2mla_s2norm_G1', 'mla_w64_care_sigma_G1', 'mla_w64_ours_unrope_cost_G1')):
    launch.run("p410m_gc", "run", BRANCH=branch, TEST_TOKENS=TEST_TOKENS)

In [ ]:
# @title KL to the teacher and to the original model, at t = 0 and the end of training, beside the paper's
launch.kl_table("p410m_klgc", TEST_TOKENS)
launch.kl_table("p410m_gc", TEST_TOKENS)